In [0]:
sdf=spark.table("predictivemaintenancecatalog.gold_data.all_features")
display(sdf)

In [0]:
sdf.columns

In [0]:
from pyspark.sql import functions as F

exclude_cols = [
    "UDI", "Product_ID",
    "TWF", "HDF", "PWF", "OSF", "RNF",
    '_ingest_timestamp',
    '_source',
    '_source_file',
    'failure_mode',
    'Mechanical_power_W',
    'Temperature_difference_K',
    'Torque_wear',
    'Speed_wear',
    'Power_wear'
]

automl_sdf= sdf.drop(*exclude_cols)

display(automl_sdf.limit(5))

In [0]:
automl_sdf.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("predictivemaintenancecatalog.gold_data.automl_original_features_binary_classification")

## Connect experiment to notebook

In [0]:
import mlflow

# Experiment renamed/moved 2026-10-22 — reference by stable ID
# (id 3358912254446233 = 00_Machine_failure_exp, shared with notebook 21)
mlflow.set_experiment(experiment_id="3358912254446233")

print("Active experiment:", mlflow.get_experiment("3358912254446233").name)


## Check avaliable libraries

In [0]:
%pip install xgboost
%pip install lightgbm xgboost

In [0]:
import sklearn
import mlflow
import importlib.util

print("scikit-learn:", sklearn.__version__)
print("MLflow:", mlflow.__version__)

for package in ["optuna", "sklearn", "xgboost", "lightgbm"]:
    available = importlib.util.find_spec(package) is not None
    print(f"{package}: {'Available' if available else 'Not installed'}")

In [0]:
display(
    automl_sdf.groupBy("Machine_failure")
    .count()
    .orderBy("Machine_failure")
)

In [0]:
print(automl_sdf.columns)

## Split the data 

In [0]:
from sklearn.model_selection import train_test_split

# Convert the historical Spark dataframe to pandas
pdf = automl_sdf.toPandas()

# Separate predictors and target
X = pdf.drop(columns=["Machine_failure"])
y = pdf["Machine_failure"].astype(int)

# Stratified split
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)

print("\nTraining failure distribution:")
print(y_train.value_counts(normalize=True))

print("\nValidation failure distribution:")
print(y_val.value_counts(normalize=True))

In [0]:
## Launch baseline model

Identify numerical and categorical features

In [0]:
# Identify numerical and categorical features

numerical_features = X_train.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

## Build the preprocessing pipeline
We will use three scikit-learn components:
- SimpleImputer — handles missing data.
- StandardScaler — standardizes numerical features.
- OneHotEncoder — converts categorical features into numerical representations.

In [0]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Numerical preprocessing
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully.")

## Create model pipelines

In [0]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, HistGradientBoostingClassifier, GradientBoostingClassifier)
from sklearn.tree import DecisionTreeClassifier, plot_tree
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

models = {
    "Logistic Regression": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        ))
    ]),

    "Decision Tree": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]),


    "Hist Gradient Boosting": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", HistGradientBoostingClassifier(
            max_iter= 200,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "Light Gradient Boosting": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", LGBMClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced",
        )),
    ]),

      #weights = compute_sample_weight(class_weight="balanced", y=y_train)  
    "XGBoost": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", XGBClassifier(
            random_state=42
        ))
    ])
}

print("Created", len(models), "model pipelines.")

## Make Data Transformation

In [0]:
X_train_transformed = preprocessor.fit_transform(X_train)

print("Original shape:", X_train.shape)
print("Transformed shape:", X_train_transformed.shape)

print("\nTransformed feature names:")
print(preprocessor.get_feature_names_out())

## Launch baseline model

### Train models and track experiments with MLflow

#### Initialize MLfow

In [0]:

import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np

print("MLflow version:", mlflow.__version__)

#### Make function to evaluate mode and make matrix


In [0]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

def evaluate_model(model, X, y):

    # Predict class labels
    y_pred = model.predict(X)

    # Predict probability of machine failure
    y_prob = model.predict_proba(X)[:, 1]

    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(
        y, y_pred, labels=[0, 1]
    ).ravel()

    metrics = {
        "accuracy": accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, zero_division=0),
        "recall": recall_score(y, y_pred, zero_division=0),
        "f1": f1_score(y, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y, y_prob),
        "pr_auc": average_precision_score(y, y_prob),
        "true_positives": int(tp),
        "false_positives": int(fp),
        "true_negatives": int(tn),
        "false_negatives": int(fn)
    }

    return metrics

#### Train all six models and log results

In [0]:
models.items()

In [0]:
# Prevent duplicate automatic logging
mlflow.sklearn.autolog(disable=True)

results = []

for model_name, pipeline in models.items():

    print(f"\nTraining: {model_name}")

    with mlflow.start_run(run_name=model_name):

        # Train on training data only
        pipeline.fit(X_train, y_train)

        # Evaluate training and validation performance
        train_metrics = evaluate_model(
            pipeline, X_train, y_train
        )

        val_metrics = evaluate_model(
            pipeline, X_val, y_val
        )

        # Log model information
        mlflow.set_tag("model_name", model_name)
        mlflow.set_tag("stage", "baseline")
        mlflow.set_tag("target", "Machine_failure")

        # Log classifier hyperparameters
        classifier = pipeline.named_steps["classifier"]

        params = classifier.get_params()

        mlflow.log_params({
            key: str(value)
            for key, value in params.items()
            if value is not None
        })

        # Log training metrics
        mlflow.log_metrics({
            f"train_{key}": value
            for key, value in train_metrics.items()
        })

        # Log validation metrics
        mlflow.log_metrics({
            f"val_{key}": value
            for key, value in val_metrics.items()
        })

        # Save results for comparison
        results.append({
            "Model": model_name,
            **{
                f"Train_{key}": value
                for key, value in train_metrics.items()
            },
            **{
                f"Val_{key}": value
                for key, value in val_metrics.items()
            }
        })

    print(f"Validation PR-AUC: {val_metrics['pr_auc']:.4f}")
    print(f"Validation Recall: {val_metrics['recall']:.4f}")
    print(f"Validation F1: {val_metrics['f1']:.4f}")

print("\nAll six models trained successfully.")

#### Compare performance

In [0]:
results_df = pd.DataFrame(results)

comparison_cols = [
    "Model",
    "Train_pr_auc",
    "Val_pr_auc",
    "Val_roc_auc",
    "Val_precision",
    "Val_recall",
    "Val_f1",
    "Val_accuracy"
]

comparison_df = (
    results_df[comparison_cols]
    .sort_values("Val_pr_auc", ascending=False)
    .reset_index(drop=True)
)

display(comparison_df)

In [0]:
%pip install seaborn

In [0]:
import mlflow
from mlflow.tracking import MlflowClient
import matplotlib.pyplot as plt
import numpy as np

# The experiment is set by ID in cell 6 — use the same ID here so
# renaming/moving the experiment can never break this cell
EXPERIMENT_ID = "3358912254446233"
print("Using experiment:", mlflow.get_experiment(EXPERIMENT_ID).name, "(id:", EXPERIMENT_ID + ")")

client = MlflowClient()
all_runs = client.search_runs([EXPERIMENT_ID], order_by=["attributes.start_time DESC"])

# Only keep runs that logged confusion-matrix metrics — this experiment is
# shared with notebook 21 and also holds threshold_selected runs that use
# different metric names and would crash the plot below
required_metrics = {
    "val_true_negatives", "val_false_positives",
    "val_false_negatives", "val_true_positives",
    "val_f1", "val_recall",
}

# Cell 33 may have been executed more than once — keep only the
# newest run per model so each appears once in the plot
runs, seen = [], set()
for run in all_runs:
    if not required_metrics.issubset(run.data.metrics):
        continue
    name = run.data.tags.get("model_name", run.info.run_name)
    if name not in seen:
        seen.add(name)
        runs.append(run)

runs.sort(key=lambda r: r.data.metrics.get("val_f1", 0), reverse=True)

n = len(runs)
ncols = min(3, n)
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.5 * nrows), squeeze=False)

for i, run in enumerate(runs):
    m = run.data.metrics
    cm = np.array([
        [m["val_true_negatives"],  m["val_false_positives"]],
        [m["val_false_negatives"],  m["val_true_positives"]],
    ])
    name = run.data.tags.get("model_name", run.info.run_name)

    ax = axes[i // ncols][i % ncols]
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1], ["Pred Healthy", "Pred failure"])
    ax.set_yticks([0, 1], ["Actual Healthy", "Actual failure"])
    ax.set_title(f"{name}\nval_f1 = {m['val_f1']:.3f}  val_recall = {m['val_recall']:.3f}")
    for r in range(2):
        for c in range(2):
            ax.text(c, r, f"{cm[r, c]:,.0f}", ha="center", va="center",
                    color="white" if cm[r, c] > cm.max() / 2 else "black",
                    fontsize=13, fontweight="bold")

# hide any unused subplots
for j in range(n, nrows * ncols):
    axes[j // ncols][j % ncols].set_visible(False)

fig.suptitle(f"Validation confusion matrices — experiment {EXPERIMENT_ID}", fontsize=14)
plt.tight_layout()
plt.show()

## Tune Baseline _model_

### Create a model-specific configuration function

In [0]:
from sklearn.ensemble import HistGradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

def create_classifier(trial, model_name):

    # Common hyperparameter names can have different effects
    # across algorithms, so each model gets its own search space.

    if model_name == "Hist Gradient Boosting":

        params = {
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "max_iter": trial.suggest_int("max_iter", 50, 400),
            "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 7, 63),
            "max_depth": trial.suggest_int("max_depth", 3, 12),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 100),
            "l2_regularization": trial.suggest_float("l2_regularization", 1e-4, 10.0, log=True),
            "max_features": trial.suggest_float("max_features", 0.4, 1.0)
        }

        failure_weight = trial.suggest_float("failure_weight", 1.0, 50.0)

        return HistGradientBoostingClassifier(
            **params,
            class_weight={0: 1.0, 1: failure_weight},
            random_state=42
        )

    elif model_name == "Light Gradient Boosting":

        params = {
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "n_estimators": trial.suggest_int("n_estimators", 50, 400),
            "num_leaves": trial.suggest_int("num_leaves", 7, 63),
            "max_depth": trial.suggest_int("max_depth", 3, 12),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
            "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 50.0),
	        "feature_fraction":trial.suggest_float("feature_fraction", 0.4, 1.0)

        }

        return LGBMClassifier(
            **params,
            random_state=42,
            verbosity=-1,
            n_jobs=1
        )

    elif model_name == "XGBoost":

        params = {
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "n_estimators": trial.suggest_int("n_estimators", 100, 400),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 20.0),
            "subsample": trial.suggest_float("subsample", 0.4, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
            "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 50.0)
        }

        return XGBClassifier(
            **params,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=42,
            n_jobs=1
        )

    else:
        raise ValueError(f"Unknown model: {model_name}")

### Create one Common Optuna objective Function

In [0]:
%pip install optuna

In [0]:
import optuna

from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=3,shuffle=True,random_state=42)

def objective(trial, model_name):
    classifier = create_classifier(trial, model_name)
    pipeline = Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("classifier", classifier)
    ])
    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="average_precision",
        n_jobs=1,
        error_score="raise"
    )

    return scores.mean()

In [0]:
from optuna.samplers import TPESampler
import mlflow

model_names = [
    "Hist Gradient Boosting",
    "Light Gradient Boosting",
    "XGBoost"
]

studies = {}
tuning_results = []

for model_name in model_names:
    print(f"\nOptimizing {model_name}")

    study = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=42),
        study_name=f"{model_name}_Optimization"
    )

    with mlflow.start_run(
        run_name=f"{model_name}_Optuna"
    ):

        study.optimize(
            lambda trial: objective(trial, model_name),
            n_trials=20,
            n_jobs=1
        )

        mlflow.log_metric(
            "best_cv_average_precision",
            study.best_value
        )

        mlflow.log_params(study.best_params)

        mlflow.set_tags({
            "model_name": model_name,
            "optimization": "Optuna",
            "cv_folds": "3",
            "scoring": "average_precision"
        })

    studies[model_name] = study

    tuning_results.append({
        "Model": model_name,
        "Best_CV_AP": study.best_value,
        "Best_Trial": study.best_trial.number
    })

print("\nOptimization finished.")

### Compare the optimized results

In [0]:
import pandas as pd

tuning_df = pd.DataFrame(tuning_results)

tuning_df = tuning_df.sort_values(
    "Best_CV_AP",
    ascending=False
)

display(tuning_df)

### Refit best models

In [0]:
optimized_models = {}

for model_name, study in studies.items():

    classifier = create_classifier(
        study.best_trial,
        model_name
    )

    pipeline = Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("classifier", classifier)
    ])

    pipeline.fit(X_train, y_train)

    optimized_models[model_name] = pipeline

print("All optimized models trained.")

### Compare baseline vs. optimized models

In [0]:
from sklearn.metrics import fbeta_score
import pandas as pd

results = []

for name in optimized_models:

    for version, model in [
        ("Baseline", models[name]),
        ("Optimized", optimized_models[name])
    ]:

        metrics = evaluate_model(
            model, X_val, y_val
        )

        y_pred = model.predict(X_val)

        results.append({
            "Model": name,
            "Version": version,
            "AP": metrics["pr_auc"],
            "ROC_AUC": metrics["roc_auc"],
            "Precision": metrics["precision"],
            "Recall": metrics["recall"],
            "F1": metrics["f1"],
            "F2": fbeta_score(
                y_val,
                y_pred,
                beta=2,
                zero_division=0
            ),
            "False_Negatives": metrics["false_negatives"],
            "False_Positives": metrics["false_positives"]
        })

comparison_df = pd.DataFrame(results)

display(
    comparison_df.sort_values(
        ["Model", "Version"]
    )
)

### Evaluate optimized models and log to MLflow

In [0]:
from sklearn.metrics import fbeta_score
import mlflow
import mlflow.sklearn

# Prevent duplicate automatic logging
mlflow.sklearn.autolog(disable=True)

optimized_results = []

for model_name, pipeline in optimized_models.items():

    print(f"\nTraining optimized: {model_name}")

    with mlflow.start_run(run_name=f"{model_name}_Optimized"):

        # Train on training data only
        pipeline.fit(X_train, y_train)

        # Evaluate training and validation performance
        train_metrics = evaluate_model(
            pipeline, X_train, y_train
        )

        val_metrics = evaluate_model(
            pipeline, X_val, y_val
        )

        # Add F2-score
        train_metrics["f2"] = fbeta_score(
            y_train,
            pipeline.predict(X_train),
            beta=2,
            zero_division=0
        )

        val_metrics["f2"] = fbeta_score(
            y_val,
            pipeline.predict(X_val),
            beta=2,
            zero_division=0
        )

        # Log model information
        mlflow.set_tag("model_name", model_name)
        mlflow.set_tag("stage", "optimized")
        mlflow.set_tag("target", "Machine_failure")
        mlflow.set_tag("optimization_method", "Optuna")

        # Log classifier hyperparameters
        classifier = pipeline.named_steps["classifier"]

        params = classifier.get_params()

        mlflow.log_params({
            key: str(value)
            for key, value in params.items()
            if value is not None
        })

        # Log training metrics
        mlflow.log_metrics({
            f"train_{key}": value
            for key, value in train_metrics.items()
        })

        # Log validation metrics
        mlflow.log_metrics({
            f"val_{key}": value
            for key, value in val_metrics.items()
        })

        # Record Optuna cross-validation performance
        mlflow.log_metric(
            "best_cv_average_precision",
            studies[model_name].best_value
        )

        # Save results for comparison
        optimized_results.append({
            "Model": model_name,
            **{
                f"Train_{key}": value
                for key, value in train_metrics.items()
            },
            **{
                f"Val_{key}": value
                for key, value in val_metrics.items()
            }
        })

    print(f"Validation PR-AUC: {val_metrics['pr_auc']:.4f}")
    print(f"Validation Recall: {val_metrics['recall']:.4f}")
    print(f"Validation F1: {val_metrics['f1']:.4f}")
    print(f"Validation F2: {val_metrics['f2']:.4f}")

print("\nAll optimized models evaluated successfully.")

### Classification Threshold Optimization

#### Select the models

In [0]:
candidate_models = {}

for name in [
    "Hist Gradient Boosting",
    "Light Gradient Boosting",
    "XGBoost"
]:
    candidate_models[f"{name} - Baseline"] = models[name]
    candidate_models[f"{name} - Optimized"] = optimized_models[name]

print("Models selected:", len(candidate_models))

#### Evaluate different thresholds

In [0]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    fbeta_score
)

thresholds = np.unique(
    np.concatenate([
        np.logspace(-6, -2, 100),
        [0.01, 0.02, 0.03, 0.05, 0.10, 0.50]
    ])
)

threshold_results = []

for model_name, model in candidate_models.items():
    y_prob = model.predict_proba(X_val)[:, 1]
    for threshold in thresholds:
        y_pred = (y_prob >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(
            y_val,
            y_pred,
            labels=[0, 1]
        ).ravel()
        threshold_results.append({
            "Model": model_name,
            "Threshold": threshold,
            "Recall": recall_score(y_val, y_pred),
            "Precision": precision_score(
                y_val, y_pred, zero_division=0
            ),
            "F2": fbeta_score(
                y_val, y_pred, beta=2, zero_division=0
            ),
            "TP": tp,
            "FN": fn,
            "FP": fp,
            "TN": tn,
            "Alert_Rate": (tp + fp) / len(y_val)
        })

threshold_df = pd.DataFrame(threshold_results)

display(threshold_df)

#### Find the best threshold for each model

In [0]:
target_recall = 0.90

best_thresholds = []

for model_name, group in threshold_df.groupby("Model"):

    eligible = group[group["Recall"] >= target_recall]

    if not eligible.empty:
        best = eligible.sort_values(
            ["FP", "F2"],
            ascending=[True, False]
        ).iloc[0]
    else:
        best = group.sort_values(
            ["Recall", "FP"],
            ascending=[False, True]
        ).iloc[0]

    best_thresholds.append(best)

best_thresholds_df = (
    pd.DataFrame(best_thresholds)
    .sort_values(["FN", "FP"])
    .reset_index(drop=True)
)

display(best_thresholds_df)

#### Visualize recall versus threshold

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))

sns.lineplot(
    data=threshold_df,
    x="Threshold",
    y="Recall",
    hue="Model"
)

plt.axhline(
    0.95,
    linestyle="--",
    color="red",
    label="95% recall target"
)

plt.title("Machine Failure Recall vs Classification Threshold")
plt.xlabel("Classification Threshold")
plt.ylabel("Recall")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### save the opimized models

#### Model configuration

In [0]:
model_configs = {
    "XGBoost - Optimized": {
        "pipeline": optimized_models["XGBoost"],
        "threshold": 0.10
    },
    "Light Gradient Boosting - Baseline": {
        "pipeline": models["Light Gradient Boosting"],
        "threshold": 0.009111627561154896
    },
    "Hist Gradient Boosting - Baseline": {
        "pipeline": models["Hist Gradient Boosting"],
        "threshold": 0.01
    },
    "Light Gradient Boosting - Optimized": {
        "pipeline": optimized_models["Light Gradient Boosting"],
        "threshold": 0.03
    },
    "Hist Gradient Boosting - Optimized": {
        "pipeline": optimized_models["Hist Gradient Boosting"],
        "threshold": 0.01
    },
    "XGBoost - Baseline": {
        "pipeline": models["XGBoost"],
        "threshold": 0.00628029144183426
    }
}

#### Define the thresholds

In [0]:
model_configs = {
    "XGBoost - Optimized": {
        "pipeline": optimized_models["XGBoost"],
        "threshold": 0.10
    },
    "Light Gradient Boosting - Baseline": {
        "pipeline": models["Light Gradient Boosting"],
        "threshold": 0.009111627561154896
    },
    "Hist Gradient Boosting - Baseline": {
        "pipeline": models["Hist Gradient Boosting"],
        "threshold": 0.01
    },
    "Light Gradient Boosting - Optimized": {
        "pipeline": optimized_models["Light Gradient Boosting"],
        "threshold": 0.03
    },
    "Hist Gradient Boosting - Optimized": {
        "pipeline": optimized_models["Hist Gradient Boosting"],
        "threshold": 0.01
    },
    "XGBoost - Baseline": {
        "pipeline": models["XGBoost"],
        "threshold": 0.00628029144183426
    }
}

#### Create a model wrapper that applies the threshold

In [0]:
import mlflow
import numpy as np
import pandas as pd

class ThresholdClassifier(mlflow.pyfunc.PythonModel):

    def __init__(self, pipeline, threshold):
        self.pipeline = pipeline
        self.threshold = float(threshold)

    def predict(self, context, model_input, params=None):

        # Calculate scores for machine failure (class 1)
        probabilities = self.pipeline.predict_proba(
            model_input
        )[:, 1]

        # Apply the model-specific threshold
        predictions = (
            probabilities >= self.threshold
        ).astype(int)

        return pd.DataFrame({
            "failure_probability": probabilities,
            "predicted_failure": predictions
        })

#### Save all six models to MLflow

In [0]:
from mlflow.models import infer_signature
from sklearn.metrics import (
    confusion_matrix,
    recall_score,
    precision_score,
    fbeta_score
)

mlflow.sklearn.autolog(disable=True)

saved_models = {}

input_example = X_train.iloc[:5].copy()

for model_name, config in model_configs.items():

    pipeline = config["pipeline"]
    threshold = config["threshold"]

    print(f"\nSaving: {model_name}")

    # Create threshold-aware model
    wrapped_model = ThresholdClassifier(
        pipeline=pipeline,
        threshold=threshold
    )

    # Evaluate at selected threshold
    y_prob = pipeline.predict_proba(X_val)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val, y_pred, labels=[0, 1]
    ).ravel()

    # Infer output schema
    prediction_example = wrapped_model.predict(
        None, input_example
    )

    signature = infer_signature(
        input_example,
        prediction_example
    )

    with mlflow.start_run(
        run_name=f"{model_name}_Threshold_Final"
    ) as run:

        mlflow.log_params({
            "model_name": model_name,
            "classification_threshold": threshold
        })

        mlflow.log_metrics({
            "val_recall": recall_score(y_val, y_pred),
            "val_precision": precision_score(
                y_val, y_pred, zero_division=0
            ),
            "val_f2": fbeta_score(
                y_val, y_pred,
                beta=2,
                zero_division=0
            ),
            "val_TP": int(tp),
            "val_FN": int(fn),
            "val_FP": int(fp),
            "val_TN": int(tn),
            "val_alert_rate": float(y_pred.mean())
        })

        mlflow.set_tags({
            "stage": "threshold_selected",
            "target": "Machine_failure",
            "threshold_applied": "true"
        })

        model_info = mlflow.pyfunc.log_model(
            name=f"threshold_model_{model_name}" ,
            python_model=wrapped_model,
            input_example=input_example,
            signature=signature,
            pip_requirements=[
                f"mlflow=={mlflow.__version__}",
                f"scikit-learn=={__import__('sklearn').__version__}",
                f"pandas=={pd.__version__}",
                f"numpy=={np.__version__}",
                f"lightgbm=={__import__('lightgbm').__version__}",
                f"xgboost=={__import__('xgboost').__version__}",
                "cloudpickle"
            ]
        )

        saved_models[model_name] = model_info.model_uri

        print("Saved:", model_info.model_uri)

print("\nAll six models saved successfully.")

#### Verify that a saved model applies its threshold

In [0]:
# Load model from MLflow
loaded_model = mlflow.pyfunc.load_model(
    saved_models["XGBoost - Optimized"]
)

# Predict on validation observations
predictions = loaded_model.predict(X_val)

display(predictions.head(10))

In [0]:
original_predictions = (
    optimized_models["XGBoost"]
    .predict_proba(X_val)[:, 1] >= 0.10
).astype(int)

saved_predictions = predictions[
    "predicted_failure"
].to_numpy()

assert np.array_equal(
    original_predictions,
    saved_predictions
)

print("Saved model predictions verified!")

In [0]:
2+2